# Preparación de datos

**Conjunto de datos:** Dataset 7 - Matriz Empresa - Variable

**Nombre de archivo:** air_permits_matrix.json

## 0. Inicialización

Instalar ydata-profiling

In [1]:
!pip install ydata-profiling

Importaciones

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from ydata_profiling import ProfileReport
import seaborn as sns

import folium
import json
from tabulate import tabulate

Visualización de tablas y gráficas

In [3]:
sns.set_style("darkgrid")

def print_table(df):
    print(tabulate(df, headers='keys', tablefmt='simple_outline'))

Lectura y muestra del archivo

In [4]:
permits = pd.read_csv('../data/preparada/emission_permits.csv')
fuel_matrix = pd.read_csv('../data/enriquecida/air_fuel_matrix.csv')
source_matrix = pd.read_csv('../data/enriquecida/air_source_matrix.csv')

**1.** Eliminar columnas

In [5]:
permits = permits[['ID', 'TipoCombustible', 'TipoFuenteEmision']]
permits

,ID,TipoCombustible,TipoFuenteEmision
0,1,Otros,Horno
1,2,Carbón,Caldera horno
2,3,ACPM,Caldera horno
3,4,Fuel Oil No.8,Planta de asfalto
4,5,Carbón,Caldera horno
...,...,...,...
531,540,Sin definir,Sin definir
532,541,Sin definir,Sin definir
533,542,Sin definir,Sin definir
534,543,Sin definir,Sin definir


**12.** Obtener probabilidades de incidencia de las empresas

In [6]:
# Nos aseguramos de que los nombres de columnas coincidan
fuel_matrix = fuel_matrix.rename(columns={'Tipo de combustible': 'TipoCombustible'})
source_matrix = source_matrix.rename(columns={'Tipo de fuente': 'TipoFuenteEmision'})

# Lista para almacenar los resultados
resultados = []

# Iteramos sobre cada variable
for var in fuel_matrix['Variable'].unique():
    # Subconjuntos para esta variable específica
    fuel_sub = fuel_matrix[fuel_matrix['Variable'] == var][['TipoCombustible', 'Probabilidad', 'Ponderación']]
    src_sub = source_matrix[source_matrix['Variable'] == var][['TipoFuenteEmision', 'Probabilidad', 'Ponderación']]
    
    # Para cada fila en df original
    for idx, row in permits.iterrows():
        tipo_combustible = row['TipoCombustible']
        tipo_fuente = row['TipoFuenteEmision']
        id = row['ID']
        
        # Obtener probabilidades y ponderaciones
        prob_fuel = fuel_sub[fuel_sub['TipoCombustible'] == tipo_combustible]['Probabilidad'].values
        pond_fuel = fuel_sub[fuel_sub['TipoCombustible'] == tipo_combustible]['Ponderación'].values
        
        prob_source = src_sub[src_sub['TipoFuenteEmision'] == tipo_fuente]['Probabilidad'].values
        pond_source = src_sub[src_sub['TipoFuenteEmision'] == tipo_fuente]['Ponderación'].values
        
        # Calcular probabilidad de emisión si existen los valores
        if len(prob_fuel) > 0 and len(prob_source) > 0:
            probabilidad_emision = (
                (prob_fuel[0] * pond_fuel[0] + prob_source[0] * pond_source[0]) / 9
            )
        else:
            probabilidad_emision = None
        
        # Agregar el resultado
        resultados.append({
            'IDEmpresa': id,
            'Variable': var,
            'ProbabilidadEmision': probabilidad_emision
        })

# Crear el dataframe final despivoteado
df = pd.DataFrame(resultados)

**13.** Ordenar por ID y variable

In [7]:
df.sort_values(by=['IDEmpresa', 'Variable'], inplace=True)

Resultado final

In [8]:
df

,IDEmpresa,Variable,ProbabilidadEmision
1608,1,CO,0.577778
2144,1,NO,0.711111
1072,1,NO2,0.711111
3216,1,NOX,0.755556
2680,1,PM10,0.800000
...,...,...,...
1607,544,NO2,0.266667
3751,544,NOX,0.311111
3215,544,PM10,0.422222
1071,544,PM2.5,0.355556


In [9]:
reporte = ProfileReport(df)
reporte.to_file("archivos_generados/Reporte perfilamiento - Dataset 7.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:00<00:00, 410.40it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Exportar a CSV

In [10]:
df.to_csv('../data/preparada/air_permits_matrix.csv', index=False)